# DataPipeline Activity Scanner

Scans all DataPipeline items across one or more Microsoft Fabric workspaces
and produces a flat table of every activity's metadata.

## What it does
- Lists every DataPipeline in each target workspace via the Fabric Items API
- Fetches each pipeline's JSON definition via `getDefinition`
- Expands the `activities` array into one row per activity
- Emits sentinel rows for empty pipelines and failed definition fetches
  so the output is a complete workspace inventory

## Output columns
`WorkspaceId`, `WorkspaceName`, `PipelineId`, `PipelineName`, `PipelineDescription`,
`ActivityIndex`, `ActivityName`, `ActivityType`, `ActivityDescription`,
`DependsOn`, `DependencyConditions`, `TimeoutMinutes`, `RetryCount`,
`RetryIntervalSeconds`, `PipelineURL`, `zUPD`

## Limitations
- Requires at least Contributor access on each workspace; Viewer-only principals
  cannot call `getDefinition` and will produce `[ERROR: ...]` rows.
- `DependsOn` and `DependencyConditions` are flattened; the per-dependency
  condition mapping is not preserved in the flat output.
- `ActivityIndex` reflects position in the JSON `activities` array, not execution
  order (which is determined by the `dependsOn` graph).
- Trial workspaces or workspaces without a Fabric capacity SKU may not support
  `getDefinition`.

## Version: 1.0.0
## Last updated: 2026-05-20

# Libraries

In [ ]:
# No additional pip installs required — sempy.fabric, polars, and pytz are
# pre-installed in Microsoft Fabric Spark runtimes.
# %pip install -q semantic-link polars pytz

import base64
import json
import polars as pl
import sempy.fabric as fabric
import notebookutils as nbutl

from datetime import datetime
from typing import Any
import pytz

# Parameters

In [ ]:
# List of workspaces to scan — supports both workspace ID (recommended) and workspace name
workspaceList = ['YOUR_WORKSPACE_NAME_OR_ID']

# Destination Delta Table path (abfss://... path).
# Leave as empty string '' to skip the Delta write and display results only.
destTablePath = ''  # e.g. 'abfss://{workspaceId}@onelake.dfs.fabric.microsoft.com/{lakehouseId}/Tables/{schema}/{table}'

# Local timezone for the zUPD audit column
# See https://en.wikipedia.org/wiki/List_of_tz_database_time_zones
timezone_LCL = 'Australia/Adelaide'

# Configuration

In [ ]:
# Datetime watermark — captured once so every row in a run shares the same zUPD
zUPD_UTC, zUPD_LCL = datetime.now(), datetime.now(pytz.timezone(timezone_LCL))

# Fabric portal base URL — used to construct PipelineURL
FABRIC_PORTAL_BASE = 'https://app.powerbi.com/groups'

# Functions

In [ ]:
# -------------------- Utilities -------------------- #

def parse_timeout_to_minutes(timeout_str: str | None) -> float | None:
    """
    Convert an ADF-style timeout string to total minutes.

    ADF timeout format is 'D.HH:MM:SS' or 'HH:MM:SS'.
    Returns None if the input is absent or cannot be parsed.

    Args:
        timeout_str: Timeout string from activity policy, e.g. '0.01:30:00'.

    Returns:
        Total minutes as a float, or None.

    Example:
        >>> parse_timeout_to_minutes('0.01:30:00')
        90.0
        >>> parse_timeout_to_minutes('00:45:00')
        45.0
    """
    if not timeout_str:
        return None
    try:
        if '.' in timeout_str:
            day_part, time_part = timeout_str.split('.', 1)
            days = int(day_part)
        else:
            days = 0
            time_part = timeout_str
        h, m, s = time_part.split(':')
        total_minutes = days * 1440 + int(h) * 60 + int(m) + int(s) / 60
        return round(total_minutes, 4)
    except Exception:
        return None


def build_pipeline_url(workspace_id: str, pipeline_id: str) -> str:
    """
    Construct the Fabric portal URL for a DataPipeline item.

    Args:
        workspace_id: Workspace GUID.
        pipeline_id: DataPipeline item GUID.

    Returns:
        Full portal URL string.
    """
    return f'{FABRIC_PORTAL_BASE}/{workspace_id}/pipelines/{pipeline_id}'


# -------------------- API Wrappers -------------------- #

def get_workspace_pipelines(client: Any, workspace_id: str) -> list[dict]:
    """
    Fetch all DataPipeline items from a workspace, handling pagination.

    Args:
        client: An authenticated FabricRestClient instance.
        workspace_id: Workspace GUID.

    Returns:
        A list of item dicts from the Fabric Items API.

    Example:
        >>> client = fabric.FabricRestClient()
        >>> pipelines = get_workspace_pipelines(client, 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx')
    """
    pipelines = []
    url = f'/v1/workspaces/{workspace_id}/items?type=DataPipeline'

    while url:
        response = client.get(url).json()
        pipelines.extend(response.get('value', []))
        continuation = response.get('continuationToken')
        url = (
            f'/v1/workspaces/{workspace_id}/items?type=DataPipeline&continuationToken={continuation}'
            if continuation else None
        )

    return pipelines


def get_pipeline_definition(client: Any, workspace_id: str, pipeline_id: str) -> dict:
    """
    Fetch and decode a DataPipeline's JSON definition via getDefinition.

    Args:
        client: An authenticated FabricRestClient instance.
        workspace_id: Workspace GUID.
        pipeline_id: DataPipeline item GUID.

    Returns:
        The decoded pipeline properties dict (containing 'activities', etc.).

    Raises:
        RuntimeError: If the pipeline-content.json part is not found in the response.

    Example:
        >>> defn = get_pipeline_definition(client, ws_id, pl_id)
        >>> activities = defn.get('activities', [])
    """
    resp = client.post(f'/v1/workspaces/{workspace_id}/items/{pipeline_id}/getDefinition')
    resp.raise_for_status()
    parts = resp.json().get('definition', {}).get('parts', [])

    for part in parts:
        if part.get('path') == 'pipeline-content.json':
            raw = base64.b64decode(part['payload']).decode('utf-8')
            return json.loads(raw).get('properties', {})

    raise RuntimeError('pipeline-content.json part not found in getDefinition response')


# -------------------- Record Builder -------------------- #

def build_activity_records(
    pipeline: dict,
    workspace_id: str,
    workspace_name: str,
    activities: list[dict],
    error_msg: str | None = None,
) -> list[dict]:
    """
    Convert a list of activity dicts into flat record dicts for LazyFrame ingestion.

    Emits one row per activity, a single empty-pipeline sentinel row when
    activities is empty, or a single error sentinel row when error_msg is set.
    Sentinel rows have ActivityIndex=None to allow easy downstream filtering.

    Args:
        pipeline: Raw item dict from the Fabric Items API.
        workspace_id: Workspace GUID.
        workspace_name: Workspace display name.
        activities: Parsed list of activity objects (may be empty).
        error_msg: If set, emit one error sentinel row instead of real rows.

    Returns:
        A list of flat record dicts.
    """
    base = {
        'WorkspaceId': workspace_id,
        'WorkspaceName': workspace_name,
        'PipelineId': pipeline['id'],
        'PipelineName': pipeline['displayName'],
        'PipelineDescription': pipeline.get('description', ''),
        'PipelineURL': build_pipeline_url(workspace_id, pipeline['id']),
    }

    sentinel = {
        **base,
        'ActivityIndex': None,
        'ActivityName': f'[ERROR: {error_msg}]' if error_msg else None,
        'ActivityType': None,
        'ActivityDescription': None,
        'DependsOn': None,
        'DependencyConditions': None,
        'TimeoutMinutes': None,
        'RetryCount': None,
        'RetryIntervalSeconds': None,
    }

    if error_msg is not None or not activities:
        return [sentinel]

    records = []
    for idx, act in enumerate(activities):
        depends_on_entries = act.get('dependsOn', [])
        upstream_names = [d.get('activity', '') for d in depends_on_entries]
        all_conditions = [c for d in depends_on_entries for c in d.get('dependencyConditions', [])]
        policy = act.get('policy', {})

        records.append({
            **base,
            'ActivityIndex': idx,
            'ActivityName': act.get('name'),
            'ActivityType': act.get('type'),
            'ActivityDescription': act.get('description'),
            'DependsOn': ', '.join(upstream_names) if upstream_names else None,
            'DependencyConditions': ', '.join(all_conditions) if all_conditions else None,
            'TimeoutMinutes': parse_timeout_to_minutes(policy.get('timeout')),
            'RetryCount': policy.get('retry'),
            'RetryIntervalSeconds': policy.get('retryIntervalInSeconds'),
        })
    return records


# -------------------- Main Scanner -------------------- #

# Explicit schema so pl.LazyFrame works correctly even when all_records is empty
_ACTIVITY_SCHEMA = {
    'WorkspaceId': pl.String,
    'WorkspaceName': pl.String,
    'PipelineId': pl.String,
    'PipelineName': pl.String,
    'PipelineDescription': pl.String,
    'ActivityIndex': pl.Int32,
    'ActivityName': pl.String,
    'ActivityType': pl.String,
    'ActivityDescription': pl.String,
    'DependsOn': pl.String,
    'DependencyConditions': pl.String,
    'TimeoutMinutes': pl.Float64,
    'RetryCount': pl.Int32,
    'RetryIntervalSeconds': pl.Int32,
    'PipelineURL': pl.String,
}


def scan_pipeline_activities(
    workspace: str,
    update_time: datetime = zUPD_LCL,
) -> pl.LazyFrame:
    """
    Scan all DataPipeline items in a workspace and return a Polars LazyFrame
    with one row per activity (or sentinel rows for empty/errored pipelines).

    Args:
        workspace: Workspace name or ID.
        update_time: Timestamp to stamp on every row as zUPD.

    Returns:
        A pl.LazyFrame following the output schema.

    Example:
        >>> lf = scan_pipeline_activities('my-workspace')
        >>> df = lf.collect()
        >>> display(df)
    """
    client = fabric.FabricRestClient()
    workspace_name, workspace_id = fabric.resolve_workspace_name_and_id(workspace)

    pipelines = get_workspace_pipelines(client, workspace_id)
    print(f'[{workspace_name}] Found {len(pipelines)} DataPipeline(s)')

    all_records: list[dict] = []

    for pipeline in pipelines:
        pipeline_name = pipeline['displayName']
        try:
            defn = get_pipeline_definition(client, workspace_id, pipeline['id'])
            activities = defn.get('activities', [])
            records = build_activity_records(pipeline, workspace_id, workspace_name, activities)
            print(f'  [{pipeline_name}] {len(activities)} activity/activities')
        except Exception as exc:
            error_msg = str(exc)[:200]
            print(f'  [{pipeline_name}] ERROR: {error_msg}')
            records = build_activity_records(
                pipeline, workspace_id, workspace_name, activities=[], error_msg=error_msg
            )

        all_records.extend(records)

    return (
        pl.LazyFrame(all_records, schema=_ACTIVITY_SCHEMA)
        .with_columns(pl.lit(update_time).dt.replace_time_zone('UTC').alias('zUPD'))
        .select([
            'WorkspaceId', 'WorkspaceName',
            'PipelineId', 'PipelineName', 'PipelineDescription',
            'ActivityIndex', 'ActivityName', 'ActivityType', 'ActivityDescription',
            'DependsOn', 'DependencyConditions',
            'TimeoutMinutes', 'RetryCount', 'RetryIntervalSeconds',
            'PipelineURL', 'zUPD',
        ])
        .sort(['WorkspaceName', 'PipelineName', 'ActivityIndex'])
    )

# Main

In [ ]:
# Scan all workspaces and collect into a single DataFrame — collect exactly once
df_pipeline_activities = (
    pl.concat([scan_pipeline_activities(w) for w in workspaceList], how='vertical')
    .collect()
)

print(f'\nTotal rows: {len(df_pipeline_activities)}')

# Evaluation

In [ ]:
display(df_pipeline_activities)

# Export

In [ ]:
# Write to Delta Table — skipped if destTablePath is empty
if destTablePath and destTablePath.strip():
    df_pipeline_activities.write_delta(
        destTablePath,
        mode='overwrite',
#        delta_write_options={'schema_mode': 'overwrite'}
    )
    print(f'Written to: {destTablePath}')
else:
    print('destTablePath is empty — Delta write skipped.')